# Instructor Demo: Pandas Deep Dive (pandas 3+)

**Week 3 Day 1, instructor-facing.** One organized tour of the pandas concepts this course relies on, from beginner to advanced. Use it live: run a cell, ask the room what will happen first, then run the next.

Teaching arc: **this notebook (pandas) -> Polars translation (Activity 2) -> engine comparison at scale (DataFrame Engines bonus lab)**. Everything here reappears later this week as SQL in BigQuery, so keep naming the bridges.

| Level | Sections |
|---|---|
| Beginner | 1 to 6: structures, reading data, inspecting, selecting, filtering, sorting |
| Intermediate | 7 to 13: cleaning, types, strings, new columns, Copy-on-Write, groupby, joins |
| Advanced | 14 to 19: reshaping, time series, window operations, memory, chaining, Arrow |

Kernel: the shared repo-root `.venv` (pandas 3.0.x in the classroom). Every cell also runs on pandas 2.3 for portability; a guard cell below aligns the behavior.

In [3]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.3
numpy: 2.5.1


## What Changed in pandas 3 (talk track)

Released January 2026. Three headliners, all visible in this demo:

1. **Strings are real strings.** Text columns now infer as `str` dtype (Arrow-backed when PyArrow is installed), not the old `object` dtype. Cleaner dtypes, less memory, faster string ops.
2. **Copy-on-Write everywhere.** Any slice or selection behaves as a copy. Chained assignment (`df[df.a > 0]["b"] = 1`) no longer works at all, and `SettingWithCopyWarning` is gone because the ambiguity is gone. Section 11 demos this.
3. **`pd.col` expressions.** A lambda-free way to refer to columns inside `assign` and friends: `df.assign(total=pd.col("fare") + pd.col("tip"))`. Section 10 demos this, and it is a perfect bridge to Polars, which is built entirely on this idea.

Also: pandas 3 requires Python 3.11+, and lots of long-deprecated APIs were removed. The upgrade rule for old code: get it warning-free on pandas 2.3 first.

In [4]:
# Portability guard: on pandas 2.x, opt in to Copy-on-Write so this notebook
# behaves exactly like pandas 3. On pandas 3 this is already the only behavior.
if pd.__version__.startswith("2"):
    pd.set_option("mode.copy_on_write", True)
    print("pandas 2 detected: Copy-on-Write enabled to match pandas 3 behavior")
else:
    print("pandas 3: Copy-on-Write is built in")

pandas 3: Copy-on-Write is built in


---
# BEGINNER

## 1. Series and DataFrame

pandas has two core structures. A **Series** is one labeled column. A **DataFrame** is a table of Series sharing one index. Everything else in pandas is operations on these two.

In [5]:
fares = pd.Series([18.5, 22.0, 16.25], name="fare_amount")
fares

0    18.50
1    22.00
2    16.25
Name: fare_amount, dtype: float64

In [6]:
trips = pd.DataFrame(
    {
        "trip_id": ["T001", "T002", "T003"],
        "borough": ["Manhattan", "Manhattan", "Queens"],
        "fare_amount": [18.5, 22.0, 16.25],
    }
)
trips

,trip_id,borough,fare_amount
0,T001,Manhattan,18.50
1,T002,Manhattan,22.00
2,T003,Queens,16.25


In [7]:
# pandas 3: note the dtype of the text columns. str, not object.
trips.dtypes

trip_id            str
borough            str
fare_amount    float64
dtype: object

## 2. Reading Data

The demo dataset is the same 16-row messy taxi CSV students use in Activity 2, so everything you show here is something they immediately reuse. The path check keeps the notebook runnable from either the repo root or this folder.

In [8]:
from pathlib import Path

candidates = [
    Path("Week 3/Labs/Day 1/data/taxi_trip_review.csv"),      # opened from repo root
    Path("../Labs/Day 1/data/taxi_trip_review.csv"),           # opened from Instructor Notes
    Path("data/taxi_trip_review.csv"),                         # copied next to the notebook
]
DATA_PATH = next(p for p in candidates if p.exists())
print("using:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
df.head()

using: ../Labs/Day 1/data/taxi_trip_review.csv


,trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
0,T001,VTS,Manhattan,Queens,18.50,3.7,5.2,2026-07-06T08:15:00Z,card
1,T002,CMT,Manhattan,Brooklyn,22.00,0.0,7.1,2026-07-06T08:40:00Z,cash
2,T003,VTS,Queens,Manhattan,16.25,2.5,4.4,2026-07-06T09:05:00Z,card
3,T004,CMT,Bronx,Manhattan,28.00,4.2,9.0,2026-07-06T09:20:00Z,card
4,T005,VTS,Brooklyn,Queens,14.00,1.5,3.2,2026-07-06T10:00:00Z,card


Useful `read_csv` knobs to mention, not necessarily demo: `usecols` (read fewer columns), `dtype` (declare types up front), `parse_dates`, `nrows` (peek at a huge file), `na_values` (what counts as missing). The same file also loads from S3/GCS URLs and from parquet with `read_parquet`, which students meet in the bonus lab.

## 3. Inspecting: the First Five Minutes with Any Dataset

Profile before touching anything. Each of these answers one question.

In [9]:
df.shape          # how much data?

(16, 9)

In [10]:
df.info()         # columns, non-null counts, dtypes, memory

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   trip_id          16 non-null     str    
 1   vendor           16 non-null     str    
 2   pickup_borough   15 non-null     str    
 3   dropoff_borough  16 non-null     str    
 4   fare_amount      16 non-null     str    
 5   tip_amount       15 non-null     float64
 6   trip_distance    16 non-null     float64
 7   pickup_ts        16 non-null     str    
 8   payment_type     16 non-null     str    
dtypes: float64(2), str(7)
memory usage: 2.0 KB


In [11]:
df.describe()     # numeric distributions; spot the impossible values

,tip_amount,trip_distance
count,15.000000,16.000000
mean,2.173333,4.700000
std,1.610109,3.538926
min,-1.000000,-4.000000
25%,1.250000,3.025000
50%,2.000000,4.950000
75%,3.350000,6.150000
max,5.000000,12.200000


In [12]:
df.describe(include="all").loc[["count", "unique", "top"]]   # text columns too

,trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
count,16,16,15,16,16,15.0,16.0,16,16
unique,15,2,5,4,14,NaN,NaN,15,2
top,T001,VTS,Manhattan,Queens,18.50,NaN,NaN,2026-07-06T08:15:00Z,card


In [13]:
df.isna().sum()   # what is missing, per column

trip_id            0
vendor             0
pickup_borough     1
dropoff_borough    0
fare_amount        0
tip_amount         1
trip_distance      0
pickup_ts          0
payment_type       0
dtype: int64

In [14]:
df["pickup_borough"].value_counts(dropna=False)   # category counts, including NaN

pickup_borough
Manhattan        6
Queens           3
Brooklyn         3
Bronx            2
Staten Island    1
NaN              1
Name: count, dtype: int64

Pause point: `describe()` shows a negative `trip_distance` min and `value_counts` shows a missing borough. The data told us where the cleaning work is before we wrote a single cleaning line.

## 4. Selecting Data

Three tools, three meanings. `[]` for columns, `.loc` for label-based rows and columns, `.iloc` for position-based. pandas 3 note: Series `[]` indexing is now strictly label-based; use `.iloc` when you mean positions.

In [15]:
df["fare_amount"].head(3)              # one column -> Series

0    18.50
1    22.00
2    16.25
Name: fare_amount, dtype: str

In [16]:
df[["trip_id", "fare_amount"]].head(3)  # list of columns -> DataFrame

,trip_id,fare_amount
0,T001,18.50
1,T002,22.00
2,T003,16.25


In [17]:
df.loc[0:3, ["trip_id", "pickup_borough"]]   # label-based, end inclusive

,trip_id,pickup_borough
0,T001,Manhattan
1,T002,Manhattan
2,T003,Queens
3,T004,Bronx


In [18]:
df.iloc[0:3, 0:2]                       # position-based, end exclusive

,trip_id,vendor
0,T001,VTS
1,T002,CMT
2,T003,VTS


## 5. Filtering

A boolean mask is a Series of True/False aligned to the rows. Build the mask, look at it once, then use it. Combine with `&` and `|`, always with parentheses.

One trap on purpose: `df.dtypes` shows `fare_amount` arrived as TEXT, because one row contains `abc`. Try `df["fare_amount"] > 15` and pandas refuses: you cannot compare text to numbers. So the numeric filters below use `trip_distance`, which parsed cleanly, and we repair `fare_amount` properly in Section 7. Let the room feel this: dirty types block analysis before cleaning.

In [19]:
mask = df["pickup_borough"] == "Manhattan"
df[mask]

,trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
0,T001,VTS,Manhattan,Queens,18.50,3.7,5.2,2026-07-06T08:15:00Z,card
1,T002,CMT,Manhattan,Brooklyn,22.00,0.0,7.1,2026-07-06T08:40:00Z,cash
6,T007,VTS,Manhattan,Manhattan,9.50,1.0,1.4,not-a-date,card
10,T011,VTS,Manhattan,Bronx,24.50,3.0,-4.0,2026-07-06T12:30:00Z,card
13,T001,VTS,Manhattan,Queens,18.50,3.7,5.2,2026-07-06T08:15:00Z,card
14,T014,CMT,Manhattan,Queens,17.00,2.0,4.9,2026-07-06T14:00:00Z,card


In [20]:
df[(df["pickup_borough"] == "Manhattan") & (df["payment_type"] == "card")]

,trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
0,T001,VTS,Manhattan,Queens,18.50,3.7,5.2,2026-07-06T08:15:00Z,card
6,T007,VTS,Manhattan,Manhattan,9.50,1.0,1.4,not-a-date,card
10,T011,VTS,Manhattan,Bronx,24.50,3.0,-4.0,2026-07-06T12:30:00Z,card
13,T001,VTS,Manhattan,Queens,18.50,3.7,5.2,2026-07-06T08:15:00Z,card
14,T014,CMT,Manhattan,Queens,17.00,2.0,4.9,2026-07-06T14:00:00Z,card


In [21]:
df[df["pickup_borough"].isin(["Bronx", "Staten Island"])]

,trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
3,T004,CMT,Bronx,Manhattan,28.00,4.20,9.0,2026-07-06T09:20:00Z,card
7,T008,CMT,Staten Island,Manhattan,35.00,5.00,12.2,2026-07-06T11:15:00Z,card
12,T013,VTS,Bronx,Queens,21.25,2.25,6.1,2026-07-06T13:15:00Z,card


In [22]:
df[df["trip_distance"].between(4, 8)][["trip_id", "trip_distance"]]

,trip_id,trip_distance
0,T001,5.2
1,T002,7.1
2,T003,4.4
5,T006,6.3
8,T009,5.0
12,T013,6.1
13,T001,5.2
14,T014,4.9
15,T015,4.9


In [23]:
# query: the same filter as a string. Readable, and a preview of SQL WHERE.
df.query("pickup_borough == 'Manhattan' and trip_distance > 4")

,trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
0,T001,VTS,Manhattan,Queens,18.50,3.7,5.2,2026-07-06T08:15:00Z,card
1,T002,CMT,Manhattan,Brooklyn,22.00,0.0,7.1,2026-07-06T08:40:00Z,cash
13,T001,VTS,Manhattan,Queens,18.50,3.7,5.2,2026-07-06T08:15:00Z,card
14,T014,CMT,Manhattan,Queens,17.00,2.0,4.9,2026-07-06T14:00:00Z,card


## 6. Sorting and Top-N

In [24]:
# Watch closely: fare_amount is still text here, so this sort is ALPHABETICAL.
# '9.50' sorts above '35.00'. A silent wrong answer, worse than an error.
df.sort_values("fare_amount", ascending=False)[["trip_id", "fare_amount"]].head(3)

,trip_id,fare_amount
8,T009,abc
6,T007,9.50
11,T012,8.75


In [25]:
df.sort_values(["pickup_borough", "trip_distance"], ascending=[True, False]).head(6)

,trip_id,vendor,pickup_borough,dropoff_borough,fare_amount,tip_amount,trip_distance,pickup_ts,payment_type
3,T004,CMT,Bronx,Manhattan,28.00,4.20,9.0,2026-07-06T09:20:00Z,card
12,T013,VTS,Bronx,Queens,21.25,2.25,6.1,2026-07-06T13:15:00Z,card
8,T009,VTS,Brooklyn,Manhattan,abc,2.00,5.0,2026-07-06T11:40:00Z,card
4,T005,VTS,Brooklyn,Queens,14.00,1.50,3.2,2026-07-06T10:00:00Z,card
11,T012,CMT,Brooklyn,Brooklyn,8.75,0.75,1.8,2026-07-06T13:00:00Z,card
1,T002,CMT,Manhattan,Brooklyn,22.00,0.00,7.1,2026-07-06T08:40:00Z,cash


In [26]:
df.nlargest(3, "trip_distance")[["trip_id", "pickup_borough", "trip_distance"]]

,trip_id,pickup_borough,trip_distance
7,T008,Staten Island,12.2
3,T004,Bronx,9.0
1,T002,Manhattan,7.1


---
# INTERMEDIATE

## 7. Missing Data and Type Coercion

This CSV is deliberately dirty: a fare of `abc`, a timestamp of `not-a-date`, an empty tip, an empty borough. The pattern is always the same: **coerce first, so problems become visible NaN, then decide the policy.**

In [27]:
clean = df.copy()

clean["fare_amount"] = pd.to_numeric(clean["fare_amount"], errors="coerce")
clean["tip_amount"] = pd.to_numeric(clean["tip_amount"], errors="coerce")
clean["trip_distance"] = pd.to_numeric(clean["trip_distance"], errors="coerce")
clean["pickup_ts"] = pd.to_datetime(clean["pickup_ts"], errors="coerce", utc=True)

clean[["fare_amount", "tip_amount", "trip_distance", "pickup_ts"]].isna().sum()

fare_amount      1
tip_amount       1
trip_distance    0
pickup_ts        1
dtype: int64

In [28]:
# Policy examples, one line each. Say the policy out loud, not just the code.
clean["tip_amount"] = clean["tip_amount"].fillna(0.0)        # missing tip means no tip
clean = clean.dropna(subset=["fare_amount", "pickup_ts", "pickup_borough"])  # required fields
print("rows after required-field policy:", len(clean))

rows after required-field policy: 13


## 8. dtypes and Conversions

`astype` converts when you already trust the values; the `to_*` functions parse and forgive. `category` is the right dtype for low-cardinality text like boroughs and payment types: less memory, faster groupby.

In [29]:
clean["payment_type"] = clean["payment_type"].astype("category")
clean["payment_type"].dtype

CategoricalDtype(categories=['card', 'cash'], ordered=False, categories_dtype=str)

In [30]:
clean["payment_type"].cat.categories

Index(['card', 'cash'], dtype='str')

## 9. String Methods

Everything under `.str` is vectorized: one call cleans the whole column. In pandas 3 these run on the new string dtype, Arrow-backed when PyArrow is installed.

In [31]:
demo = pd.Series(["  Manhattan ", "queens", "BRONX  "], name="borough")
demo.str.strip().str.title()

0    Manhattan
1       Queens
2        Bronx
Name: borough, dtype: str

In [32]:
clean["vendor"] = clean["vendor"].str.strip().str.upper()
clean["trip_id"].str.startswith("T00").head(3)

0    True
1    True
2    True
Name: trip_id, dtype: bool

## 10. New Columns Three Ways, Including `pd.col`

Direct assignment mutates. `assign` returns a new DataFrame, which chains nicely. And pandas 3 adds `pd.col`: refer to a column by name inside expressions, no lambda. If that reminds you of Polars' `pl.col`, that is exactly the point, and it makes Activity 2's translation feel natural.

In [33]:
# Way 1: direct assignment
clean["total_amount"] = clean["fare_amount"] + clean["tip_amount"]
clean[["trip_id", "total_amount"]].head(3)

,trip_id,total_amount
0,T001,22.20
1,T002,22.00
2,T003,18.75


In [34]:
# Way 2: assign with a lambda (the pandas 2 idiom)
clean.assign(tip_rate=lambda d: (d["tip_amount"] / d["fare_amount"]).round(3))[
    ["trip_id", "tip_rate"]
].head(3)

,trip_id,tip_rate
0,T001,0.200
1,T002,0.000
2,T003,0.154


In [35]:
# Way 3: assign with pd.col (pandas 3). Guarded so the cell runs anywhere.
if hasattr(pd, "col"):
    result = clean.assign(tip_rate=(pd.col("tip_amount") / pd.col("fare_amount")).round(3))
    print(result[["trip_id", "tip_rate"]].head(3))
else:
    print("pd.col arrived in pandas 3; this kernel is", pd.__version__)

  trip_id  tip_rate
0    T001     0.200
1    T002     0.000
2    T003     0.154


Avoid `df.apply(lambda row: ..., axis=1)` for arithmetic: it runs Python row by row and is often 100x slower than the vectorized versions above. Reach for `apply` only when no vectorized form exists.

## 11. Copy-on-Write: the pandas 3 Mental Model

The old pain: "is this a view or a copy?" and the dreaded `SettingWithCopyWarning`. pandas 3 ends it with one rule: **any selection behaves as a copy; to modify, do it on the DataFrame itself with `.loc`.**

In [36]:
subset = clean[clean["pickup_borough"] == "Manhattan"]
subset.loc[:, "fare_amount"] = 0.0   # modifying the selection...

print("subset fares:", subset["fare_amount"].unique())
print("original fares still intact:", clean.loc[clean["pickup_borough"] == "Manhattan", "fare_amount"].head(3).tolist())

subset fares: [0.]
original fares still intact: [18.5, 22.0, 24.5]


In [37]:
# The RIGHT way to update the original: one .loc on the DataFrame itself.
capped = clean.copy()
capped.loc[capped["fare_amount"] > 30, "fare_amount"] = 30.0
capped["fare_amount"].max()

np.float64(30.0)

Talk track: chained assignment like `clean[clean.fare_amount > 30]["fare_amount"] = 30` did unpredictable things for a decade. In pandas 3 it simply never modifies the original, so the pattern above (`df.loc[mask, col] = value`) is the only pattern to teach.

## 12. GroupBy: Split, Apply, Combine

The single most important pattern in the course. This exact operation becomes `GROUP BY` in BigQuery on Day 3 and `group_by().agg()` in Polars this afternoon.

In [38]:
clean.groupby("pickup_borough")["fare_amount"].mean().round(2)

pickup_borough
Bronx            24.62
Brooklyn         11.38
Manhattan        20.10
Queens           15.67
Staten Island    35.00
Name: fare_amount, dtype: float64

In [39]:
# Named aggregation: production style, clean output columns.
summary = clean.groupby("pickup_borough", as_index=False).agg(
    trip_count=("trip_id", "count"),
    total_revenue=("total_amount", "sum"),
    average_fare=("fare_amount", "mean"),
)
summary.sort_values("total_revenue", ascending=False).round(2)

,pickup_borough,trip_count,total_revenue,average_fare
2,Manhattan,5,112.9,20.10
0,Bronx,2,55.7,24.62
3,Queens,3,48.5,15.67
4,Staten Island,1,40.0,35.00
1,Brooklyn,2,25.0,11.38


In [40]:
# transform: aggregate, but broadcast back to every row. Ask: when is this useful?
clean["borough_avg_fare"] = clean.groupby("pickup_borough")["fare_amount"].transform("mean")
clean["vs_borough_avg"] = (clean["fare_amount"] - clean["borough_avg_fare"]).round(2)
clean[["trip_id", "pickup_borough", "fare_amount", "vs_borough_avg"]].head(6)

,trip_id,pickup_borough,fare_amount,vs_borough_avg
0,T001,Manhattan,18.50,-1.60
1,T002,Manhattan,22.00,1.90
2,T003,Queens,16.25,0.58
3,T004,Bronx,28.00,3.38
4,T005,Brooklyn,14.00,2.62
5,T006,Queens,19.75,4.08


## 13. Combining Tables: merge and concat

`merge` is SQL JOIN. `concat` stacks. The lookup-table join below is the most common shape in warehouses: facts joined to dimensions.

In [41]:
zones = pd.DataFrame(
    {
        "borough": ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"],
        "zone_group": ["Core", "Core", "Airport-heavy", "North", "South"],
    }
)

joined = clean.merge(zones, left_on="pickup_borough", right_on="borough", how="left")
joined[["trip_id", "pickup_borough", "zone_group", "fare_amount"]].head(6)

,trip_id,pickup_borough,zone_group,fare_amount
0,T001,Manhattan,Core,18.50
1,T002,Manhattan,Core,22.00
2,T003,Queens,Airport-heavy,16.25
3,T004,Bronx,North,28.00
4,T005,Brooklyn,Core,14.00
5,T006,Queens,Airport-heavy,19.75


In [42]:
# how= matters. A quick left-vs-inner demo with a lookup that is missing Queens:
partial_zones = zones[zones["borough"] != "Queens"]
left = clean.merge(partial_zones, left_on="pickup_borough", right_on="borough", how="left")
inner = clean.merge(partial_zones, left_on="pickup_borough", right_on="borough", how="inner")
print("left keeps all trips:", len(left), "| inner drops Queens trips:", len(inner))

left keeps all trips: 13 | inner drops Queens trips: 10


In [43]:
jan = clean.head(3)
feb = clean.tail(3)
pd.concat([jan, feb], ignore_index=True)[["trip_id", "pickup_borough"]]

,trip_id,pickup_borough
0,T001,Manhattan
1,T002,Manhattan
2,T003,Queens
3,T013,Bronx
4,T001,Manhattan
5,T014,Manhattan


---
# ADVANCED

## 14. Reshaping: pivot_table and melt

Wide vs long. Analysts ask for wide; models and warehouses want long. You translate.

In [44]:
pivot = clean.pivot_table(
    index="pickup_borough",
    columns="payment_type",
    values="fare_amount",
    aggfunc="mean",
    observed=True,
).round(2)
pivot

payment_type,card,cash
pickup_borough,,
Bronx,24.62,NaN
Brooklyn,11.38,NaN
Manhattan,19.62,22.00
Queens,16.25,15.38
Staten Island,35.00,NaN


In [45]:
pivot.reset_index().melt(id_vars="pickup_borough", var_name="payment", value_name="avg_fare").dropna().head(6)

,pickup_borough,payment,avg_fare
0,Bronx,card,24.62
1,Brooklyn,card,11.38
2,Manhattan,card,19.62
3,Queens,card,16.25
4,Staten Island,card,35.00
7,Manhattan,cash,22.00


## 15. Time Series: resample and rolling

A datetime index unlocks time-based operations. `resample` is groupby over time buckets; `rolling` is a sliding window. Frequency aliases are lowercase in modern pandas: `"h"` for hour, `"D"` for day, `"ME"` for month-end.

In [46]:
rng = np.random.default_rng(7)
hourly = pd.DataFrame(
    {"rides": rng.poisson(40, 24 * 14)},
    index=pd.date_range("2026-06-01", periods=24 * 14, freq="h", tz="UTC"),
)
hourly.head(3)

,rides
2026-06-01 00:00:00+00:00,42
2026-06-01 01:00:00+00:00,45
2026-06-01 02:00:00+00:00,36


In [47]:
daily = hourly.resample("D").sum()
daily.head(7)

,rides
2026-06-01 00:00:00+00:00,941
2026-06-02 00:00:00+00:00,987
2026-06-03 00:00:00+00:00,965
2026-06-04 00:00:00+00:00,984
2026-06-05 00:00:00+00:00,893
2026-06-06 00:00:00+00:00,992
2026-06-07 00:00:00+00:00,936


In [48]:
daily["rides_7d_avg"] = daily["rides"].rolling(7).mean().round(1)
daily.tail(7)

,rides,rides_7d_avg
2026-06-08 00:00:00+00:00,997,964.9
2026-06-09 00:00:00+00:00,999,966.6
2026-06-10 00:00:00+00:00,941,963.1
2026-06-11 00:00:00+00:00,994,964.6
2026-06-12 00:00:00+00:00,955,973.4
2026-06-13 00:00:00+00:00,927,964.1
2026-06-14 00:00:00+00:00,947,965.7


## 16. Ordering Operations: shift, diff, rank, cumsum

These become SQL window functions (`LAG`, `LEAD`, `RANK`, running totals) in Week 4. Plant the vocabulary now.

In [49]:
daily["prev_day"] = daily["rides"].shift(1)          # LAG
daily["day_change"] = daily["rides"].diff()           # rides - LAG(rides)
daily["running_total"] = daily["rides"].cumsum()      # running SUM
daily["busy_rank"] = daily["rides"].rank(ascending=False)  # RANK
daily.head(7)

,rides,rides_7d_avg,prev_day,day_change,running_total,busy_rank
2026-06-01 00:00:00+00:00,941,NaN,NaN,NaN,941,10.5
2026-06-02 00:00:00+00:00,987,NaN,941.0,46.0,1928,5.0
2026-06-03 00:00:00+00:00,965,NaN,987.0,-22.0,2893,7.0
2026-06-04 00:00:00+00:00,984,NaN,965.0,19.0,3877,6.0
2026-06-05 00:00:00+00:00,893,NaN,984.0,-91.0,4770,14.0
2026-06-06 00:00:00+00:00,992,NaN,893.0,99.0,5762,4.0
2026-06-07 00:00:00+00:00,936,956.9,992.0,-56.0,6698,12.0


## 17. Memory and Performance

Two habits that matter at taxi scale (3.5 million rows in the Yellow Taxi carryover): know what memory you use, and stay vectorized.

In [50]:
clean.memory_usage(deep=True)

Index               104
trip_id             158
vendor              145
pickup_borough      208
dropoff_borough     198
fare_amount         104
tip_amount          104
trip_distance       104
pickup_ts           104
payment_type        146
total_amount        104
borough_avg_fare    104
vs_borough_avg      104
dtype: int64

In [51]:
before = df["pickup_borough"].memory_usage(deep=True)
after = df["pickup_borough"].astype("category").memory_usage(deep=True)
print(f"borough column as str: {before:,} bytes | as category: {after:,} bytes")

borough column as str: 381 bytes | as category: 230 bytes


In [52]:
%%time
# Vectorized total over a scaled-up frame (about 1.1 million rows)
big = pd.concat([clean] * 100_000, ignore_index=True)
big["total"] = big["fare_amount"] + big["tip_amount"]
print(len(big), "rows")

1300000 rows
CPU times: user 2.04 s, sys: 73.7 ms, total: 2.12 s
Wall time: 2.13 s


In [53]:
%%time
# The same thing with a row-wise apply on just one tenth of the data. Compare wall times.
sample = big.head(len(big) // 10).copy()
sample["total"] = sample.apply(lambda r: r["fare_amount"] + r["tip_amount"], axis=1)
print(len(sample), "rows via apply")

130000 rows via apply
CPU times: user 397 ms, sys: 24.6 ms, total: 421 ms
Wall time: 423 ms


## 18. Method Chaining: Pipelines that Read Top to Bottom

Under Copy-on-Write, chains are cheap and safe, and a chain reads like a recipe. This is also exactly how Polars code is shaped, so students who learn this style get Polars almost for free.

In [54]:
report = (
    pd.read_csv(DATA_PATH)
    .assign(
        fare_amount=lambda d: pd.to_numeric(d["fare_amount"], errors="coerce"),
        tip_amount=lambda d: pd.to_numeric(d["tip_amount"], errors="coerce").fillna(0.0),
        pickup_ts=lambda d: pd.to_datetime(d["pickup_ts"], errors="coerce", utc=True),
    )
    .dropna(subset=["fare_amount", "pickup_ts", "pickup_borough"])
    .query("fare_amount > 0")
    .assign(total=lambda d: d["fare_amount"] + d["tip_amount"])
    .groupby("pickup_borough", as_index=False)
    .agg(trips=("trip_id", "count"), revenue=("total", "sum"))
    .sort_values("revenue", ascending=False)
    .round(2)
)
report

,pickup_borough,trips,revenue
2,Manhattan,5,112.9
0,Bronx,2,55.7
3,Queens,3,48.5
4,Staten Island,1,40.0
1,Brooklyn,2,25.0


## 19. Under the Hood: Arrow, and the Bridge Out of pandas

pandas 3's default string dtype is Arrow-backed when PyArrow is installed, and `read_parquet`/`to_parquet` are PyArrow underneath. Arrow is the shared in-memory format that lets pandas, Polars, DuckDB, and BigQuery clients pass data around without copying row by row.

Zero-copy handoff to Polars, to close the loop for this afternoon:

In [55]:
import polars as pl

pl_df = pl.from_pandas(report)
print(type(pl_df))
pl_df

<class 'polars.dataframe.frame.DataFrame'>


pickup_borough,trips,revenue
str,i64,f64
"""Manhattan""",5,112.9
"""Bronx""",2,55.7
"""Queens""",3,48.5
"""Staten Island""",1,40.0
"""Brooklyn""",2,25.0


---
## Wrap-Up: the Bridges to Name Out Loud

| Today's pandas | Where it returns |
|---|---|
| `pd.col` expressions | `pl.col` in Activity 2, this afternoon |
| `groupby().agg()` | Polars `group_by().agg()`, then BigQuery `GROUP BY` on Day 3 |
| `query("...")` | SQL `WHERE` |
| `merge(how=...)` | SQL joins, Day 3 |
| `shift`, `rank`, `cumsum` | SQL window functions, Week 4 |
| category dtype and memory | columnar formats and BigQuery pricing, Day 2 |
| vectorize, never row-loop | Spark DataFrames, Week 5 |

Timing guide if running the full tour: Beginner 25 min, Intermediate 35 min, Advanced 30 min. For a shorter session, the load-bearing sections are 3, 5, 7, 10, 11, 12, and 18.